# 🐱 Numerai CatBoost-Only Submission

This notebook trains and submits **ONLY CatBoost** to `jewellzilla_cat`.

**Use this when:**
- LightGBM + XGBoost were already submitted via automation
- You want to manually submit just CatBoost

**Runtime:** ~5 minutes


In [1]:
#!/usr/bin/env python3
"""
Numerai CatBoost-Only Submission
"""

import pandas as pd
import numpy as np
import gc
from pathlib import Path
from datetime import datetime
import time

from numerapi import NumerAPI
from catboost import CatBoostRegressor

# ============================================================
# CONFIGURATION
# ============================================================

# API Credentials
PUBLIC_ID = 'KY2YNIU7TAALGQRVVTHERFRG7QUJP3FJ'
SECRET_KEY = 'EIMJDP62GSPHETVXXZGLFLOOGKFBRBA6DYPJ7WWSVRMUR3OFKGPIOVVZYHWXBHP4'

# CatBoost Model ID
CATBOOST_MODEL_ID = '9e253cd6-6b6b-4178-a641-c9738f21eb11'  # jewellzilla_cat

# Data Configuration
DATA_VERSION = "v5.2"
TRAINING_FILE = f"{DATA_VERSION}/train.parquet"
LIVE_FILE = f"{DATA_VERSION}/live.parquet"

# Memory Saving Settings
SAMPLE_FRACTION = 0.08  # Use 8% of training data
MAX_FEATURES = 750     # Limit features

# Date for filenames
DATE_STR = datetime.now().strftime("%Y%m%d_%H%M%S")

# Submissions folder
SUBMISSIONS_DIR = Path("submissions")
SUBMISSIONS_DIR.mkdir(exist_ok=True)

print("="*70)
print("🐱 NUMERAI CATBOOST-ONLY SUBMISSION")
print("="*70)
print(f"\n📅 Date: {DATE_STR}")
print(f"💾 Sample fraction: {SAMPLE_FRACTION*100}%")
print(f"📊 Max features: {MAX_FEATURES}")
print(f"\n🎯 Model: CatBoost → jewellzilla_cat")



🐱 NUMERAI CATBOOST-ONLY SUBMISSION

📅 Date: 20251231_152428
💾 Sample fraction: 10.0%
📊 Max features: 1000

🎯 Model: CatBoost → jewellzilla_cat


In [2]:
# ============================================================
# ROBUST DOWNLOAD FUNCTION
# ============================================================

import requests
from tqdm import tqdm
import os

def download_with_retry(napi, filename, max_retries=10, chunk_size=1024*1024):
    """
    Download a Numerai dataset with retry and resume support.
    """
    dest_path = Path(filename)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Get the download URL from Numerai API
    query = "query($filename: String!) { dataset(filename: $filename) }"
    args = {'filename': filename}
    url = napi.raw_query(query, args)['data']['dataset']
    
    print(f"\n📥 Downloading {filename}...")
    
    for attempt in range(max_retries):
        try:
            resume_pos = 0
            mode = 'wb'
            
            if dest_path.exists():
                resume_pos = dest_path.stat().st_size
                mode = 'ab'
            
            headers = {}
            if resume_pos > 0:
                headers['Range'] = f'bytes={resume_pos}-'
            
            response = requests.get(url, headers=headers, stream=True, timeout=30)
            
            if 'content-range' in response.headers:
                total_size = int(response.headers['content-range'].split('/')[-1])
            else:
                total_size = int(response.headers.get('content-length', 0)) + resume_pos
            
            with open(dest_path, mode) as f:
                with tqdm(total=total_size, initial=resume_pos, unit='B', 
                          unit_scale=True, desc=filename.split('/')[-1]) as pbar:
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            
            final_size = dest_path.stat().st_size
            if final_size >= total_size:
                print(f"   ✅ Download complete!")
                return True
                
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"   ⚠️  Attempt {attempt + 1}/{max_retries} failed, retrying...")
                time.sleep(5)
            else:
                raise Exception(f"Failed to download {filename} after {max_retries} attempts")
    
    return False

print("✅ Download function ready")


✅ Download function ready


In [3]:
# ============================================================
# Connect to Numerai API
# ============================================================

print("\n" + "="*70)
print("🔌 Connecting to Numerai API")
print("="*70)

napi = NumerAPI(PUBLIC_ID, SECRET_KEY)
print("✅ Connected to Numerai API")



🔌 Connecting to Numerai API
✅ Connected to Numerai API


In [4]:
# ============================================================
# Download Data
# ============================================================

print("\n" + "="*70)
print("📥 Downloading Data")
print("="*70)

# Download training data if missing
if not Path(TRAINING_FILE).exists():
    print(f"\n📥 Downloading training data (~2.5 GB)...")
    download_with_retry(napi, TRAINING_FILE)
else:
    file_size = Path(TRAINING_FILE).stat().st_size / (1024**3)
    print(f"\n✅ Training data already exists ({file_size:.2f} GB)")

# Always download fresh live data
print(f"\n🔄 Downloading fresh live data...")
if Path(LIVE_FILE).exists():
    os.remove(LIVE_FILE)
download_with_retry(napi, LIVE_FILE)

print(f"\n✅ All data ready!")



📥 Downloading Data

✅ Training data already exists (2.40 GB)

🔄 Downloading fresh live data...

📥 Downloading v5.2/live.parquet...


live.parquet: 100%|███████████████████████████████████████████████████████████████| 9.46M/9.46M [00:06<00:00, 1.42MB/s]

   ✅ Download complete!

✅ All data ready!


In [5]:
# ============================================================
# Load and Prepare Training Data
# ============================================================

import pyarrow.parquet as pq

print("\n" + "="*70)
print("📂 Loading Training Data")
print("="*70)

start_time = datetime.now()

# Get column names
print(f"\n🔄 Reading column names from {TRAINING_FILE}...")
parquet_file = pq.ParquetFile(TRAINING_FILE)
all_columns = parquet_file.schema.names

# Get feature columns (limited to MAX_FEATURES)
all_features = [col for col in all_columns if col.startswith("feature_")]
feature_cols = all_features[:MAX_FEATURES]
print(f"   Found {len(all_features):,} features, using {len(feature_cols):,}")

# Columns to load: selected features + target
cols_to_load = feature_cols + ["target"]

# Calculate sample size
total_rows = parquet_file.metadata.num_rows
sample_size = int(total_rows * SAMPLE_FRACTION)
print(f"   Total rows: {total_rows:,}, sampling {sample_size:,} ({SAMPLE_FRACTION*100}%)")

# Load data
print(f"\n🔄 Loading {len(cols_to_load)} columns...")
training_data = pd.read_parquet(TRAINING_FILE, columns=cols_to_load)

# Sample
print(f"\n🔄 Sampling data...")
training_data = training_data.sample(n=sample_size, random_state=42)
gc.collect()

print(f"   ✅ Loaded {len(training_data):,} rows")

# Prepare X and y
X_train = training_data[feature_cols]
y_train = training_data["target"]

del training_data
gc.collect()

print(f"\n📊 Data Statistics:")
print(f"   X_train shape: {X_train.shape}")
print(f"   Target mean:   {y_train.mean():.6f}")

load_time = (datetime.now() - start_time).total_seconds()
print(f"\n⏱️  Data loading completed in {load_time:.1f}s")



📂 Loading Training Data

🔄 Reading column names from v5.2/train.parquet...
   Found 2,748 features, using 1,000
   Total rows: 2,746,268, sampling 274,626 (10.0%)

🔄 Loading 1001 columns...

🔄 Sampling data...
   ✅ Loaded 274,626 rows

📊 Data Statistics:
   X_train shape: (274626, 1000)
   Target mean:   0.499878

⏱️  Data loading completed in 87.4s


In [6]:
# ============================================================
# Load Live Data
# ============================================================

print("\n" + "="*70)
print("📂 Loading Live Data")
print("="*70)

print(f"\n🔄 Loading {LIVE_FILE}...")
live_data = pd.read_parquet(LIVE_FILE, columns=feature_cols)

print(f"   ✅ Loaded {len(live_data):,} rows")
print(f"   Live data shape: {live_data.shape}")



📂 Loading Live Data

🔄 Loading v5.2/live.parquet...
   ✅ Loaded 6,644 rows
   Live data shape: (6644, 1000)


In [7]:
# ============================================================
# Train CatBoost Model
# ============================================================

print("\n" + "="*70)
print("🐱 Training CatBoost Model")
print("="*70)

print("\n🔄 Training CatBoost...")
start = datetime.now()

cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=4,
    verbose=False
)
cat_model.fit(X_train, y_train)

train_time = (datetime.now() - start).total_seconds()
print(f"   ✅ Training completed in {train_time:.1f}s")

# Training correlation
cat_train_preds = cat_model.predict(X_train)
cat_train_corr = np.corrcoef(cat_train_preds, y_train)[0, 1]
print(f"   📈 Training correlation: {cat_train_corr:.4f}")

# Generate predictions
print("\n🔮 Generating predictions...")
cat_predictions = cat_model.predict(live_data[feature_cols])

# Normalize to [0, 1]
cat_predictions_norm = (cat_predictions - cat_predictions.min()) / (cat_predictions.max() - cat_predictions.min())

# Save submission
cat_filename = SUBMISSIONS_DIR / f"submission_catboost_{DATE_STR}.csv"
cat_submission = pd.DataFrame({
    "id": live_data.index,
    "prediction": cat_predictions_norm
})
cat_submission.to_csv(cat_filename, index=False)

print(f"   ✅ Saved {len(cat_submission):,} predictions to {cat_filename}")
print(f"   📊 Mean prediction: {cat_predictions_norm.mean():.6f}")



🐱 Training CatBoost Model

🔄 Training CatBoost...
   ✅ Training completed in 187.6s
   📈 Training correlation: 0.2508

🔮 Generating predictions...
   ✅ Saved 6,644 predictions to submissions\submission_catboost_20251231_152428.csv
   📊 Mean prediction: 0.459795


In [8]:
# ============================================================
# CatBoost Training Summary
# ============================================================

print("\n" + "="*70)
print("📊 CATBOOST SUMMARY")
print("="*70)

print(f"\nTraining correlation: {cat_train_corr:.4f}")
print(f"Prediction mean:      {cat_predictions_norm.mean():.6f}")
print(f"Submission file:      {cat_filename}")
print(f"\n✅ CatBoost ready for submission!")



📊 CATBOOST SUMMARY

Training correlation: 0.2508
Prediction mean:      0.459795
Submission file:      submissions\submission_catboost_20251231_152428.csv

✅ CatBoost ready for submission!


In [9]:
# ============================================================
# Submit CatBoost to Numerai
# ============================================================

print("\n" + "="*70)
print("🚀 Submitting CatBoost to Numerai")
print("="*70)

print(f"\n📤 Submitting to jewellzilla_cat...")
print(f"   Model ID: {CATBOOST_MODEL_ID}")
print(f"   File: {cat_filename}")

try:
    submission_id = napi.upload_predictions(cat_filename, model_id=CATBOOST_MODEL_ID)
    print(f"\n   ✅ SUCCESS!")
    print(f"   Submission ID: {submission_id}")
    print(f"\n🎉 CatBoost submitted successfully!")
except Exception as e:
    print(f"\n   ❌ FAILED: {e}")
    print(f"\n💡 Check your API credentials and model ID")


2025-12-31 15:29:20,875 INFO numerapi.base_api: uploading predictions...



🚀 Submitting CatBoost to Numerai

📤 Submitting to jewellzilla_cat...
   Model ID: 9e253cd6-6b6b-4178-a641-c9738f21eb11
   File: submissions\submission_catboost_20251231_152428.csv

   ✅ SUCCESS!
   Submission ID: dac2e875-4073-4d2f-8910-41f917c4e7c3

🎉 CatBoost submitted successfully!
